# 🎨🏝️👽 Plot the shoreline + habitable zone.

This notebook visualizes the cosmic shoreline in the context of the traditional habitable zone.


In [ ]:
format = 'paper'
if format == 'poster':
    annotation_font_size = 4
    figsize=(10,3)
    mapsize=None
if format == 'paper':
    annotation_font_size = 3
    figsize=(8,3)
    mapsize=None

In [ ]:
from shoreline import * 

## Load and label populations.

In [ ]:
# load organized populations (all planets!)
subset = 'all'
fluxlimit = 'any'
pops = load_organized_populations(subset=subset, fluxlimit=fluxlimit)[f'{subset}+{fluxlimit}']

In [ ]:
# load posterior and store in shoreline object (no-magma ocean!)
subset='all'
fluxlimit='no-magma'
uncertainties=True
posterior = az.from_netcdf(f'posteriors/{subset}+{fluxlimit}+uncertainties={uncertainties}+numpyro.nc')

In [ ]:
exoplanets = pops['everything']['transit']

to_label = exoplanets.relative_instellation() < 1
to_label *= exoplanets.distance() < 300*u.pc
print(list(exoplanets[to_label].name()))

In [ ]:
# define some planets we might want to annotate

annotation_options = dict(
    many=[
        "Mercury",
        "Earth",
        "Mars",
        "Jupiter",
        "Saturn",
        "Uranus",
        "Neptune",
        "Moon",
        "Pluto",
        "Eris",
        "Haumea",
        "Makemake",
        "Ceres",
        "Titan",
        "55 Cnc e",
        "TOI 561b",
        "LHS 3844 b",
        "GJ367b",
        "TOI-1685b",
        "GJ1252b",
        "GJ486b",
        "GJ1132b",
        "LTT1445Ab",
        "TOI-1468 b",
        "LHS 1140 c",
        "Trappist-1b",
        "Trappist-1c",
        "GJ 3929b",
        "LTT 3780b",
        "GJ-3929 b",
        "LTT-1445 A c",
        "LTT-1445 A b",
        "LHS-1140 b",
        "TOI-198 b",
        "TOI-406 c",
        "TOI-771 b",
        "HD 260655 c",
        "TOI-244 b",
        "LHS 1478 b",
        "Kepler-10b",
        "Kepler-78b",
        "K2-141 b",
        "L 98-59b",
        "GJ 1214b",
        "K2-18b",
        "TOI-700d",
        "TOI-700e",
        "Kepler-62e",
        "Kepler-62f",
        "L 98-59c",
        "L 98-59d",
    ]
    + [f"Trappist-1{k}" for k in "defgh"]
    + [
        "Kepler-1229 b",
        "Kepler-16 b",
        "Kepler-1649 c",
        "Kepler-1652 b",
        "Kepler-186 f",
        "Kepler-296 f",
        "Kepler-441 b",
        "LHS 1140 b",
        "LP 890-9 c",
        "TOI-201 c",
        "TOI-700 d",
        "TOI-715 b",
        "TRAPPIST-1 e",
        "TRAPPIST-1 f",
        "TRAPPIST-1 g",
        "TRAPPIST-1 h",
    ],
    few=[
        "Mercury",
        "Earth",
        "Mars",
        "Jupiter",
        "Saturn",
        "Uranus",
        "Neptune",
        "Moon",
        "Pluto",
        "Titan",
        "Ceres",
        "Kepler-1229 b",
        "Kepler-16 b",
        "Kepler-1649 c",
        "Kepler-1652 b",
        "Kepler-186 f",
        "Kepler-296 f",
        "Kepler-441 b",
        "LHS 1140 b",
        "LP 890-9 c",
        "TOI-201 c",
        "TOI-700 d",
        "TOI-715 b",
        #"TRAPPIST-1 e",
        "TRAPPIST-1 f",
    ],
)

In [ ]:
for k, v in clean_pops(pops).items():
    v.annotate_planets = True
    v.annotate_kw = dict(
        rotation=0,
        rotation_mode="anchor",
        format="   {}",
        fontsize=annotation_font_size,
    )
pops["everything"]["transit"].color = "black"

In [ ]:
luminosity_to_type_and_mass = {
1e0:'$\sf G2, 1M_\odot$',
1e-1:'$\sf K7, 0.65M_\odot$', 
1e-2:'$\sf M3, 0.3M_\odot$',
1e-3:'$\sf M6, 0.1M_\odot$'
}

In [ ]:
# decide how many planets to label
for annotation_type, planets_to_annotate in annotation_options.items():

    # set the planets to label
    for k, v in clean_pops(pops).items():
        v.annotate_kw['names']=planets_to_annotate

    m = Shoreline_SemimajorAxis_x_StellarLuminosity_x_Radius(posterior=posterior, 
                                                            sliceaxis = Radius(lim=[0.8, 1.8]*u.Rearth),
                                                            invisible_fraction=0.9) 
    g = SliceGridGallery(m, N=2, dpi=300, figsize=figsize, mapsize=mapsize, sharey=False)
    g.build(pops)
    g.refine()

    # add spectral type + mass labels
    for m in g.maps.values():
        m.ax.set_ylabel(g.maps[0].ax.get_ylabel())
        plt.sca(m.ax)
        for l, s in luminosity_to_type_and_mass.items():
            plt.text(1e-3, l, f' {s}', ha='left', va='center', fontsize='xx-small')

    g.add_panel_labels()    

    # add a colorbar for the shoreline probability
    g.maps[len(g.maps)-1].add_colorbar()
    plt.savefig(f'figures/shoreline+habitable-zone+2-panel+{annotation_type}-annotations.pdf')



In [ ]:
!cp figures/shoreline+habitable-zone+2-panel+few-annotations.pdf paper-figures/.

## Plot the habitable zone + shoreline together.

The original 3D shoreline parameter space of escape velocity, bolometric flux, stellar luminosity can be approximately transformed into planet radius (from escape velocity assuming the small-planet mass-radius relation we derived), semimajor axis (from the bolometric flux and luminosity), and the same stellar luminosity. We can compare to two estimates of the habitable zone: Earth-equivalent instellation, and the Kopparapu et al. (2013) habitable zone that accounts for M dwarf spectra being slightly easier to absorb in Earth-like atmospheres. 

In [ ]:
# decide how many planets to label
for annotation_type, planets_to_annotate in annotation_options.items():

    # set the planets to label
    for k, v in clean_pops(pops).items():
        v.annotate_kw['names']=planets_to_annotate


    m = Shoreline_SemimajorAxis_x_StellarLuminosity_x_Radius(posterior=posterior, sliceaxis = Radius(lim=[0.4, 2.0]*u.Rearth)) 
    g = SliceGridGallery(m, N=4, dpi=300)
    g.build(pops)
    g.refine()


    # add spectral type + mass labels
    for m in g.maps.values():
        m.ax.set_ylabel(g.maps[0].ax.get_ylabel())
        plt.sca(m.ax)
        for l, s in luminosity_to_type_and_mass.items():
            plt.text(1e-3, l, f' {s}', ha='left', va='center', fontsize='xx-small')
            
    g.add_panel_labels()
    
    plt.savefig(f'figures/shoreline+habitable-zone+4-panel+{annotation_type}-annotations.pdf')

In [ ]:
from exoatlas.visualizations import * 

# decide how many planets to label
for annotation_type, planets_to_annotate in annotation_options.items():

    # set the planets to label
    for k, v in clean_pops(pops).items():
        v.annotate_kw['names']=planets_to_annotate

    m = Shoreline_SemimajorAxis_x_StellarLuminosity_x_Radius(posterior=posterior, sliceaxis = Radius(lim=[0.5, 2.11]*u.Rearth)) 
    g = SliceAnimatedGallery(m, N=8, dpi=600, figsize=(5,5))
    g.animate(pops, filename=f'figures/shoreline+habitable-zone+{annotation_type}-annotations-animated.mp4', fps=2)


In [ ]:
m = Shoreline_SemimajorAxis_x_StellarLuminosity_x_Radius(posterior=posterior) # , sliceaxis=Radius(lim=[0, np.inf]*u.Rearth)
d = ErrorMap(xaxis=Depth(), yaxis=StellarLuminosity(), sliceaxis=m.plottable['slice'])
#solar.annotate_kw['rotation'] = 90
g = Gallery(maps=[m,d], figsize=(8,5))
g.build(pops)
g.maps['semimajoraxis_x_stellar_luminosity'].plot_hz_earth()
g.maps['semimajoraxis_x_stellar_luminosity'].plot_hz_kopparapu()

plt.sca(g.maps['transit_depth_x_stellar_luminosity'].ax)
L = np.logspace(-4, 2) * u.Lsun
Rs = 10**Mamajek().tofrom("logR")("logL")(np.log10(L.to_value('Lsun')))*u.Rsun
Rp = 1*u.Rearth
depth = (Rp/Rs).decompose()**2
plt.plot(depth, L, color='seagreen', alpha=0.75)
plt.xlim(0.5e-5, 2e-2)
plt.legend(frameon=False)

plt.xlabel('Transit Depth (fraction of starlight)')
plt.savefig('figures/small-stars-are-easier.pdf')

